# 01 — Explore trades

Pull a bounded window into a DataFrame and look at it. Everything here is
read-only and local; nothing writes back to Kafka.

Prerequisites are the same as `00_stream_health.ipynb`.

In [ ]:
import matplotlib.pyplot as plt

import devlab
from devlab import frames

target = devlab.resolve()
target

## Grab a window

`collect()` is bounded by **both** a message count and a wall clock, whichever
comes first — so this cell always returns, even against a dead broker.

`offset_reset="earliest"` reads what is already retained. Use `"latest"` to
watch only what arrives from now on.

In [ ]:
records = devlab.collect(target, limit=20_000, seconds=60.0, offset_reset="earliest")
df = frames.trades_frame(records)
print(f"{len(df):,} trades  {df['event_ts'].min()} .. {df['event_ts'].max()}")
df.head()

Prices and sizes cross the wire as **strings** so no precision is lost in
transit. `trades_frame` converts them to float64 for arithmetic and plotting
and keeps the exact originals in `price_str` / `size_str` — check those when a
number looks wrong.

In [ ]:
df[["venue", "instrument_id", "price", "size", "notional", "side", "latency_ms"]].describe()

## Who is producing what

In [ ]:
frames.frame(
    df.groupby(["venue", "instrument_id"], observed=True)
    .agg(trades=("trade_id", "count"), volume=("size", "sum"), notional=("notional", "sum"))
    .reset_index()
    .sort_values("notional", ascending=False)
    .to_dict("records")
)

## Price over the window

One line per instrument, each on its own axis — BTC and DOGE on a shared scale
tells you nothing.

In [ ]:
instruments = df["instrument_id"].value_counts().head(4).index.tolist()
fig, axes = plt.subplots(
    len(instruments), 1, figsize=(11, 2.4 * len(instruments)), sharex=True, squeeze=False
)
for axis, instrument in zip(axes.ravel(), instruments, strict=True):
    for venue, group in df[df["instrument_id"] == instrument].groupby("venue", observed=True):
        axis.plot(group["event_ts"], group["price"], linewidth=0.8, label=venue)
    axis.set_ylabel(instrument)
    axis.legend(loc="upper left", fontsize=8)
axes.ravel()[-1].set_xlabel("event time (UTC)")
fig.tight_layout()

## Ingest latency

`ingest_ts - event_ts`: how long between the exchange stamping the trade and
this process seeing it. Includes exchange-side delay, network, and our own
parse time, so read it as an upper bound rather than a network measurement.

A long right tail usually means the bounded queue was saturating.

In [ ]:
axis = df["latency_ms"].clip(upper=df["latency_ms"].quantile(0.99)).hist(bins=60, figsize=(11, 3))
axis.set_xlabel("ingest latency (ms, 99th percentile clipped)")
axis.set_ylabel("trades")
print(df.groupby("venue", observed=True)["latency_ms"].describe()[["50%", "90%", "max"]])

## Do the two venues agree

Same instrument, same minute, VWAP on each venue and the spread between them.
Persistent non-zero spread is normal — different venues, different order books.
A spread that trends is worth a second look.

In [ ]:
comparison = frames.venue_comparison(df, freq="1min")
comparison.tail(10)

In [ ]:
btc = comparison[comparison["instrument_id"] == "BTC-USD"]
if "spread_bps" in btc.columns and not btc.empty:
    axis = btc.plot(x="event_ts", y="spread_bps", figsize=(11, 3), legend=False)
    axis.axhline(0, color="black", linewidth=0.6)
    axis.set_ylabel("binance - coinbase (bps)")
else:
    print("need both venues in the window for a spread")